# TabICL in the browser

[TabICL](https://github.com/soda-inria/tabicl) is a **tabular foundation model**: a pretrained transformer that classifies a new table by *in-context learning*. `fit` only stores the training rows; `predict` is a single forward pass, with no per-dataset training.

Everything here runs **in your browser**: PyTorch, scikit-learn and TabICL are WebAssembly builds from [emscripten-forge](https://github.com/emscripten-forge/recipes), running in JupyterLite with the xeus-python kernel. Your data never leaves the tab.

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import torch, sklearn, tabicl
from tabicl import TabICLClassifier

print(f"torch {torch.__version__}, scikit-learn {sklearn.__version__}, tabicl {tabicl.__version__}")

## Download the pretrained model

The checkpoint (105 MB) comes straight from the [Hugging Face Hub](https://huggingface.co/jingang/TabICL) with the browser's `fetch`, and is written to the kernel's in-memory file system.

In [ ]:
import pyjs

CKPT = "tabicl-classifier-v2-20260212.ckpt"
URL = f"https://huggingface.co/jingang/TabICL/resolve/main/{CKPT}"
MODEL_PATH = f"/tmp/{CKPT}"

if not os.path.exists(MODEL_PATH):
    t = time.time()
    response = await pyjs.js.fetch(URL)
    data = pyjs.to_py(pyjs.js.Uint8Array.new(await response.arrayBuffer()))
    with open(MODEL_PATH, "wb") as f:
        f.write(bytes(data))
    print(f"downloaded {os.path.getsize(MODEL_PATH) / 2**20:.0f} MiB in {time.time() - t:.1f}s")

## Decision boundaries, no training

Three toy 2D problems. TabICL sees the training points as context and predicts the class probability on a grid.

In [ ]:
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.preprocessing import StandardScaler

datasets = {
    "moons": make_moons(n_samples=150, noise=0.25, random_state=0),
    "circles": make_circles(n_samples=150, noise=0.15, factor=0.5, random_state=1),
    "linear": make_classification(n_samples=150, n_features=2, n_redundant=0, n_informative=2,
                                  n_clusters_per_class=1, class_sep=1.2, random_state=2),
}

clf = TabICLClassifier(model_path=MODEL_PATH, n_estimators=4)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (name, (X, y)) in zip(axes, datasets.items()):
    X = StandardScaler().fit_transform(X)
    xx, yy = np.meshgrid(np.linspace(-2.8, 2.8, 40), np.linspace(-2.8, 2.8, 40))
    t = time.time()
    clf.fit(X, y)
    proba = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, proba, levels=20, cmap="RdBu_r", alpha=0.8)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu_r", edgecolors="k", s=20)
    ax.set_title(f"{name}  ({time.time() - t:.1f}s)")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("TabICL class probability, computed in the browser")
plt.tight_layout()
plt.show()

## A real table: breast cancer diagnosis

TabICL against classic scikit-learn models, out of the box, on a held-out test set.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

models = {
    "TabICL": TabICLClassifier(model_path=MODEL_PATH, n_estimators=4),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=200, n_jobs=1, random_state=0),
    "Gradient boosting": HistGradientBoostingClassifier(random_state=0),
}

rows = []
for name, model in models.items():
    t = time.time()
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    rows.append({"model": name,
                 "accuracy": accuracy_score(y_test, proba > 0.5),
                 "ROC AUC": roc_auc_score(y_test, proba),
                 "time [s]": time.time() - t})

pd.DataFrame(rows).set_index("model").style.format("{:.3f}").highlight_max(subset=["accuracy", "ROC AUC"], color="#b6e3b6")